# 감성 분석 모델 학습 및 추론
### 1. 데이터 로드
- 긍정, 부정 라벨
- 여러 감정들의 라벨
### 2. 데이터 전처리
### 3. 모델 정의 및 생성
### 4. 모델 학습
### 5. 추론

In [6]:
# !pip install pandas scikit-learn tensorflow konlpy

1. 데이터 로드

In [7]:
import pandas as pd
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

train_df = pd.read_table('https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt')
test_df = pd.read_table('https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt')

2. 데이터 전처리

In [8]:
train_df = train_df.dropna().drop_duplicates()
test_df = test_df.dropna().drop_duplicates()

train_df['document'] = train_df['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","")
train_df['document'] = train_df['document'].str.replace('^ +', "")
train_df['document'].replace('', np.nan, inplace=True)
train_df = train_df.dropna()

test_df['document'] = test_df['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","")
test_df['document'] = test_df['document'].str.replace('^ +', "")
test_df['document'].replace('', np.nan, inplace=True)
test_df = test_df.dropna()

okt = Okt()
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

X_train = []
for sentence in train_df['document']:
    tokenized_sentence = okt.morphs(sentence, stem=True)
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords]
    X_train.append(stopwords_removed_sentence)

X_test = []
for sentence in test_df['document']:
    tokenized_sentence = okt.morphs(sentence, stem=True)
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords]
    X_test.append(stopwords_removed_sentence)

y_train = np.array(train_df['label'])
y_test = np.array(test_df['label'])

vocab_size = 10000
tokenizer = Tokenizer(vocab_size)
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

max_len = 30
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_29596\1568310826.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['document'].replace('', np.nan, inplace=True)
C:\Users\Playdata\AppData\Local\Temp\ipykernel_29596\1568310826.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

3. 모델 정의 및 생성

In [9]:
embedding_dim = 100
hidden_units = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))
model.add(LSTM(hidden_units))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])

4. 모델 학습

In [10]:
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 145s 75ms/step - acc: 0.8300 - loss: 0.3780 - val_acc: 0.8564 - val_loss: 0.3340
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 127s 67ms/step - acc: 0.8721 - loss: 0.2973 - val_acc: 0.8613 - val_loss: 0.3229
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 135s 72ms/step - acc: 0.8912 - loss: 0.2571 - val_acc: 0.8626 - val_loss: 0.3293
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 144s 73ms/step - acc: 0.9077 - loss: 0.2237 - val_acc: 0.8590 - val_loss: 0.3569
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - acc: 0.9224 - loss: 0.1899 - val_acc: 0.8537 - val_loss: 0.3933


5. 추론 및 사용자 입력

In [ ]:
def sentiment_predict(new_sentence):
    new_sentence = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣 ]','', new_sentence)
    new_sentence = okt.morphs(new_sentence, stem=True)
    new_sentence = [word for word in new_sentence if not word in stopwords]
    encoded = tokenizer.texts_to_sequences([new_sentence])
    pad_new = pad_sequences(encoded, maxlen = max_len)
    
    score = float(model.predict(pad_new))
    if(score > 0.5):
        print(f"{score*100:.2f}% 확률로 초긍정 해피 리뷰임.\n")
    else:
        print(f"{(1 - score)*100:.2f}% 확률로 초부정 ㅠㅠ 리뷰임.\n")

while True:
    user_input = input("종료하려면 '끝' 입력: ")
    if user_input == '끝':
        print("분석 끝.")
        break
    sentiment_predict(user_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
53.92% 확률로 초긍정 해피 리뷰임.



C:\Users\Playdata\AppData\Local\Temp\ipykernel_29596\1630866937.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  score = float(model.predict(pad_new))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
97.52% 확률로 초부정 리뷰임.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
99.93% 확률로 초부정 리뷰임.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
86.98% 확률로 초부정 리뷰임.

분석 끝.
